# 004 · Temporary Files and Directories

> **Standard Library › Files & Paths** · Python 3.14

The `tempfile` module creates temporary files and directories **securely** (unique, unpredictable names, private permissions) and — when used as context managers — **deletes them automatically**. It is what you use in tests, for scratch data, and for atomic writes.

**In this notebook you will learn:**

- `TemporaryDirectory` — the workhorse, and how cleanup works
- `NamedTemporaryFile` (with `delete=` and the 3.12+ `delete_on_close=`) and the anonymous `TemporaryFile`
- the low-level `mkstemp` / `mkdtemp`, which you must clean up yourself
- `gettempdir`, `SpooledTemporaryFile`, and why you should never build temp names by hand

## .NET mental model

| .NET | Python |
|---|---|
| `Path.GetTempPath()` | `tempfile.gettempdir()` |
| `Path.GetTempFileName()` (creates a 0-byte file, you delete it) | `tempfile.mkstemp()` (returns fd + path, you delete it) |
| `Directory.CreateTempSubdirectory()` (.NET 7+) | `tempfile.mkdtemp()` |
| `new FileStream(path, ..., FileOptions.DeleteOnClose)` | `tempfile.NamedTemporaryFile()` / `TemporaryFile()` |
| (no built-in auto-deleting temp *directory*) | `tempfile.TemporaryDirectory()` |

The big win over .NET: Python's context-manager versions **clean up for you**, including whole directory trees.

## `TemporaryDirectory` — the one you'll use most

```csharp
// C#
var dir = Directory.CreateTempSubdirectory("demo-");
try
{
    File.WriteAllText(Path.Combine(dir.FullName, "data.txt"), "hello");
}
finally
{
    dir.Delete(recursive: true);
}
```

In Python the `with` block does the `try/finally` for you. The `as` target is the directory path as a **string**, so wrap it in `Path`.

In [ ]:
import tempfile
from pathlib import Path

with tempfile.TemporaryDirectory(prefix="demo-") as tmp_name:
    root = Path(tmp_name)
    (root / "sub").mkdir()
    (root / "sub" / "data.txt").write_text("hello", encoding="utf-8")
    print(type(tmp_name).__name__, root.name.startswith("demo-"))
    print(sorted(p.relative_to(root).as_posix() for p in root.rglob("*")))

print(root.exists())     # False — the whole tree is gone

You can also keep the object and call `cleanup()` yourself — handy in notebooks or `setUp`/`tearDown` style tests. If you forget, it is cleaned up when the object is garbage-collected (with a `ResourceWarning`), but don't rely on that.

In [ ]:
td = tempfile.TemporaryDirectory()
work = Path(td.name)
(work / "scratch.bin").write_bytes(b"\x00" * 16)
print(work.exists(), [p.name for p in work.iterdir()])

td.cleanup()
print(work.exists())

Useful options:

- `prefix=` / `suffix=` / `dir=` — control the name and parent location
- `ignore_cleanup_errors=True` (3.10+) — don't raise if cleanup fails (e.g. a file still open on Windows)
- `delete=False` (3.12+) — keep the directory after the `with` block (useful for debugging a failing test)

In [ ]:
with tempfile.TemporaryDirectory(delete=False) as kept_name:
    (Path(kept_name) / "evidence.log").write_text("look at me later", encoding="utf-8")

kept = Path(kept_name)
print(kept.exists())                # True — still there

import shutil
shutil.rmtree(kept)                 # so we clean up manually
print(kept.exists())

## `NamedTemporaryFile`

Creates a real file with a visible name (`f.name`) and returns an open file object. By default it is opened in binary mode `"w+b"`; pass `mode="w+"` and `encoding=` for text.

In [ ]:
with tempfile.NamedTemporaryFile(mode="w+", encoding="utf-8", suffix=".csv") as f:
    f.write("id,name\n1,alice\n")
    f.seek(0)                                   # same handle can read back ("+" mode)
    print(f.read(), end="")
    print(Path(f.name).suffix, Path(f.name).exists())
    name = f.name

print(Path(name).exists())    # False — deleted on close

### `delete` and `delete_on_close` (3.12+)

A common need is: *write a temp file, close it, then hand the **path** to something else* (another library, a subprocess). With the defaults that's impossible — closing the file deletes it. Python 3.12 added `delete_on_close`:

| `delete` | `delete_on_close` | File is deleted… |
|---|---|---|
| `True` (default) | `True` (default) | when the file is closed (or the `with` exits) |
| `True` | `False` | when the **`with` block exits**, *not* on `close()` |
| `False` | (ignored) | never — you delete it yourself |

In [ ]:
def count_lines(path: str) -> int:           # some API that wants a file *path*
    with open(path, encoding="utf-8") as f:
        return sum(1 for _ in f)

with tempfile.NamedTemporaryFile("w", encoding="utf-8", delete_on_close=False) as f:
    f.write("a\nb\nc\n")
    f.close()                                  # flush & close — file still exists
    print(Path(f.name).exists(), count_lines(f.name))

print(Path(f.name).exists())                   # deleted when the with block ended

⚠️ **Gotcha:** on **Windows**, a file opened by `NamedTemporaryFile` with the default settings can't be opened a second time by name while it is still open. The `delete_on_close=False` + `f.close()` pattern above is the portable way to share the file by path.

With `delete=False` you own the file — remember to remove it:

In [ ]:
import os

f = tempfile.NamedTemporaryFile("w", encoding="utf-8", delete=False, suffix=".txt")
with f:
    f.write("kept")
print(Path(f.name).read_text(encoding="utf-8"))
os.remove(f.name)                     # manual cleanup
print(Path(f.name).exists())

## `TemporaryFile` — anonymous scratch space

`TemporaryFile()` is for data that no one else needs to open by name. On POSIX systems the file is unlinked immediately, so it **has no path at all** (its `name` is just the file descriptor number) and it disappears even if the process crashes.

In [ ]:
with tempfile.TemporaryFile(mode="w+", encoding="utf-8") as f:
    for i in range(3):
        f.write(f"row {i}\n")
    f.seek(0)
    print(f.read().splitlines())
    print(type(f.name).__name__)      # int on macOS/Linux (a file descriptor), str on Windows

**Tip:** `SpooledTemporaryFile(max_size=...)` keeps data **in memory** until it grows beyond `max_size` bytes, then transparently spills to a real temp file — great for uploads of unknown size *(see Standard Library › Files & Paths › 005 · In-Memory Streams with io)*.

In [ ]:
with tempfile.SpooledTemporaryFile(max_size=1024, mode="w+", encoding="utf-8") as f:
    f.write("small payload — stays in RAM")
    f.seek(0)
    print(f.read())

## Low level: `mkstemp` and `mkdtemp`

These just **create** a uniquely named file or directory and give it back — no auto-delete. They're the building blocks the classes above are made of, and what you need when the temp file must outlive a function (e.g. the atomic write pattern in *Standard Library › Files & Paths › 002 · Reading and Writing Files*).

- `mkstemp()` → `(fd, path)`: an **open OS-level file descriptor** plus the path. Close the fd (or wrap it with `open(fd, ...)`) and delete the file yourself.
- `mkdtemp()` → path of a new directory. Delete it with `shutil.rmtree`.

In [ ]:
fd, path = tempfile.mkstemp(suffix=".json", prefix="cfg-")
try:
    with open(fd, "w", encoding="utf-8") as f:     # open() adopts the fd and closes it
        f.write('{"debug": true}')
    print(Path(path).name.startswith("cfg-"), Path(path).read_text(encoding="utf-8"))
finally:
    os.remove(path)

d = tempfile.mkdtemp(prefix="job-")
try:
    print(Path(d).is_dir(), Path(d).name.startswith("job-"))
finally:
    shutil.rmtree(d)
print(Path(path).exists(), Path(d).exists())

⚠️ **Gotcha:** forgetting to close the descriptor from `mkstemp` leaks a file handle — `fd` is a plain `int`, so nothing closes it for you. `os.close(fd)` if you only want the path.

## Where do temp files go? `gettempdir()`

`tempfile.gettempdir()` returns the directory used by default — chosen from the `TMPDIR`, `TEMP`, `TMP` environment variables, then platform defaults — like `Path.GetTempPath()`. All functions accept `dir=` to override it per call (handy for keeping the temp file on the same drive as the destination).

In [ ]:
temp_root = Path(tempfile.gettempdir())
print(temp_root.is_dir(), temp_root.is_absolute())    # the path itself is machine-specific

with tempfile.TemporaryDirectory() as parent:
    with tempfile.NamedTemporaryFile(dir=parent) as f:
        print(Path(f.name).parent == Path(parent))    # True

## Security: never invent temp names yourself

Code like `open("/tmp/myapp_" + str(os.getpid()), "w")` is vulnerable: another user on the same machine can **predict** the name and pre-create a file or symlink there (a classic *symlink attack* / race condition), redirecting your writes. `tempfile`:

- uses random, unpredictable names,
- creates the file atomically with "must not exist" semantics (like `FileMode.CreateNew`),
- gives it owner-only permissions (`0o600` for files, `0o700` for directories on POSIX).

`tempfile.mktemp()` (only returns a name, without creating anything) is **deprecated** for exactly this reason — don't use it.

In [ ]:
fd, path = tempfile.mkstemp()
os.close(fd)
print(oct(os.stat(path).st_mode & 0o777))    # 0o600 on macOS/Linux: owner read/write only
os.remove(path)

## Summary

| Task | C# / .NET | Python |
|---|---|---|
| Temp folder location | `Path.GetTempPath()` | `tempfile.gettempdir()` |
| Auto-deleted temp dir | manual `try/finally` + `Delete(true)` | `with tempfile.TemporaryDirectory() as d:` |
| Keep temp dir for debugging | — | `TemporaryDirectory(delete=False)` (3.12+) |
| Auto-deleted named file | `FileOptions.DeleteOnClose` | `NamedTemporaryFile()` |
| Close but keep until block ends | — | `NamedTemporaryFile(delete_on_close=False)` (3.12+) |
| Anonymous scratch file | `FileOptions.DeleteOnClose` | `TemporaryFile()` |
| Unique file, manual cleanup | `Path.GetTempFileName()` | `fd, path = tempfile.mkstemp()` |
| Unique dir, manual cleanup | `Directory.CreateTempSubdirectory()` | `tempfile.mkdtemp()` |

## Exercises

1. Using `TemporaryDirectory`, create three files `a.txt`, `b.txt`, `c.txt` containing their own names, then print the concatenated content of all three (sorted by name). Prove the directory is gone afterwards.
2. Write a function `write_temp_report(lines: list[str]) -> str` that writes the lines to a `NamedTemporaryFile` with `delete=False` and suffix `.txt`, and returns the path. Call it, print the file's content, then delete the file.
3. Use `NamedTemporaryFile(delete_on_close=False)` to write `"42"`, close it, and read it back with `Path.read_text` inside the `with` block.

In [ ]:
# Your solutions here

<details>
<summary>Show solutions</summary>

```python
import os
import tempfile
from pathlib import Path

# 1 — TemporaryDirectory + Path
with tempfile.TemporaryDirectory() as d:
    root = Path(d)
    for name in ["c.txt", "a.txt", "b.txt"]:
        (root / name).write_text(name, encoding="utf-8")
    print("".join(p.read_text(encoding="utf-8") for p in sorted(root.glob("*.txt"))))
print(root.exists())   # False

# 2 — delete=False: the caller owns the file
def write_temp_report(lines: list[str]) -> str:
    with tempfile.NamedTemporaryFile("w", encoding="utf-8", suffix=".txt", delete=False) as f:
        f.writelines(line + "\n" for line in lines)
        return f.name

path = write_temp_report(["total: 3", "failed: 0"])
print(Path(path).read_text(encoding="utf-8"), end="")
os.remove(path)

# 3 — delete_on_close=False: closing doesn't delete, leaving the with block does
with tempfile.NamedTemporaryFile("w", encoding="utf-8", delete_on_close=False) as f:
    f.write("42")
    f.close()
    print(Path(f.name).read_text(encoding="utf-8"))
print(Path(f.name).exists())   # False
```

</details>